In [38]:
import pandas as pd
pd.set_option('display.max_rows', 50)

In [39]:
roster_dates = {
    # "Sep 11, 2023": 230054,
    "Aug 11, 2023": 230050,
    "Jul 17, 2023": 230045,
    "Jun 19, 2023": 230040,
    "May 16, 2023": 230034,
    "Apr 17, 2023": 230028,
    "Mar 24, 2023": 230021,
    "Feb 22, 2023": 230015,
    "Jan 18, 2023": 230010,
    "Dec 17, 2022": 230008,
    "Nov 16, 2022": 230006,
    "Oct 7, 2022": 230004,
    "Sep 1, 2022": 230001
}


# Individual Data

## Merging

In [40]:
df = pd.DataFrame()
for date, roster in roster_dates.items():
    temp_df = pd.read_csv(f"../../data/player_data_train/players_{roster}.csv", index_col=False)
    df = pd.concat([df, temp_df], ignore_index=True)
    

## Basic Preprocess

In [41]:
import re

def _clean_sofifa_data( df):
    df.columns = df.columns.str.replace(" / ", "_").str.replace(" & ", "_").str.replace(" ", "_")

    df['height'] = df["height"].str.split("cm").str[0].astype(int)
    df['weight'] = df["weight"].str.split("kg").str[0].astype(int) 

    df['value'] = df['value'].apply(parse_monetary_value)
    df['wage'] = df['wage'].apply(parse_monetary_value)
    df['release_clause'] = df['release_clause'].apply(parse_monetary_value)
    df['team'] = df['team_contract'].apply(extract_team_name)

def parse_monetary_value(row): 
    row = row.split('€')[1]
    if "M" in row:
        value = float(row.replace("M", "")) * 1_000_000
    elif "K" in row:
        value = float(row.replace("K", "")) * 1_000
    else:
        value = float(row)

    return value

def extract_team_name(row):
    """Extract team name from 'Team & Contract' column by splitting on first 4-digit pattern"""
    match = re.search(r'\d{4}', str(row))
    if match:
        # Get position of first 4-digit pattern
        position = match.start()
        # Extract everything before it and strip whitespace
        team_name = row[:position].strip()
        return team_name
    else:
        # If no 4-digit pattern found, return original value
        return row

def extract_primary_position(row):
    if "," in row:
        row = row.split(",")[0]
    
    return row.strip()

def extract_name_from_position(row):
    pattern = r'[A-Z]{2,}'
    match = re.search(pattern, row)
    if match:
        location = match.start()
        name = row[:location].strip()
        
        return name
    else:
        return "HELLO"

df.columns = df.columns.str.strip().str.lower()

df['name'] = df['name'].apply(extract_name_from_position)
df["name"] = df['name'].str.lower()      

_clean_sofifa_data(df)


In [42]:
position_map = {
    'GK': 'GK',
    'CB': 'DF', 'LB': 'DF', 'RB': 'DF', 'LWB': 'DF', 'RWB': 'DF',
    'CDM': 'MF', 'CM': 'MF', 'CAM': 'MF', 'LM': 'MF', 'RM': 'MF',
    'ST': 'FW', 'CF': 'FW', 'LW': 'FW', 'RW': 'FW'
}

df['general_position'] = df['best_position'].map(position_map)

In [43]:
# Rearrange columns: prioritize key identifiers first, then metadata
priority_cols = ['id', 'name', 'birth_year', 'general_position', 'team', 'roster_date', 'season_code', 'weight', 'value', 'wage']
other_cols = [col for col in df.columns if col not in priority_cols]
df = df[priority_cols + other_cols]

In [44]:
def format_date(row_name):
    # Convert to datetime with error handling
    df[row_name] = pd.to_datetime(df[row_name], format='%b %d, %Y', errors='coerce')
format_date(row_name="roster_date")
format_date(row_name="joined")

## Features

In [45]:
def change_dtype(row):
    if "+" in str(row):
        values = row.split("+")
        row = int(values[0]) + int(values[1])
    elif "-" in str(row):
        values = row.split("-")
        row = int(values[0]) - int(values[1])
    else:
        row = int(row)
    
    return row

In [46]:
# This cell just redefines stat_columns as provided (already defined above, but repeated here as requested)
stat_columns = [
    'overall_rating', 'potential', "crossing", "finishing", "heading_accuracy",
    "short_passing", "volleys", "dribbling", "curve", "fk_accuracy",
    "long_passing", "ball_control", "acceleration", "sprint_speed", "agility",
    "reactions", "balance", "shot_power", "jumping", "stamina", "strength",
    "long_shots", "aggression", "interceptions", "attack_position", "vision",
    "penalties", "composure", "defensive_awareness", "standing_tackle",
    "sliding_tackle", "gk_diving", "gk_handling", "gk_kicking",
    "gk_positioning", "gk_reflexes"
]

In [47]:
df[stat_columns] = df[stat_columns].applymap(change_dtype)

/var/folders/0_/_3dg0b615p38ptjrygv5db6h0000gn/T/ipykernel_55452/4190917171.py:1: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df[stat_columns] = df[stat_columns].applymap(change_dtype)


In [48]:
df.foot = df.foot.map({"Right":0, "Left":1})
df.attacking_work_rate = df.attacking_work_rate.map({"High":2, "Medium":1, "Low":0})
df.defensive_work_rate = df.defensive_work_rate.map({"High":2, "Medium":1, "Low":0})

In [49]:
dropping = []
for key, value in df.isna().sum().items():
    if value != 0:
        dropping.append(key)
    else:
        pass

df.drop(dropping, axis=1, inplace=True)
df.drop(['team_contract', 'real_face', 'body_type', "best_position", "club_position"], axis=1, inplace=True)
df.drop_duplicates(inplace=True)

## Feature mapping
foot: {"Right":0, "Left":1}
attacking_work_rate: {"High":2, "Medium":1, "Low":0}
defensive_work_rate: {"High":2, "Medium":1, "Low":0}

# Match Data

In [50]:
df_match = pd.read_csv("../../data/match_data_train/match_data.csv", index_col=False)
df_match.drop(["Unnamed: 0", "match_url"], axis=1, inplace=True)

In [51]:


# Function to clean and extract month from messy date format
def extract_month_from_date(date_str):
    """Extract month from date string like 'May 2604:00 AM'"""
    if pd.isna(date_str):
        return None
    
    import re
    
    # Extract month names using regex
    month_pattern = r'^(Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)'
    match = re.search(month_pattern, str(date_str))
    
    if match:
        return match.group(1)
    else:
        return None

# Apply extraction
df_match['month'] = df_match['date'].apply(extract_month_from_date)
df_match.drop(['date'], axis=1, inplace=True)

# Check results
print("Month extraction results:")
print(df_match['month'].value_counts())

Sample date values:
['May 2912:30 AM', 'May 2912:30 AM', 'May 2912:30 AM', 'May 2912:30 AM', 'May 2912:30 AM', 'May 2912:30 AM', 'May 2912:30 AM', 'May 2912:30 AM', 'May 2912:30 AM', 'May 2912:30 AM']

Unique date patterns:
date
May 2101:30 AM    11
May 2912:30 AM    10
Jun 0504:00 AM    10
28.05. 21:30       9
21.05. 21:30       9
Name: count, dtype: int64
Month extraction results:
month
Oct    360
Apr    334
May    324
Aug    272
Sep    239
Feb    214
Jan    201
Mar    190
Nov    181
Jul    132
Jun     81
Dec     75
Name: count, dtype: int64


In [52]:
def assign_year_from_month_and_league(df_match):
    """Assign year based on month and league season"""
    
    def get_year(row):
        month = row['month']
        league = row['league']
        
        if pd.isna(month):
            return None
        
        # Season mappings
        season_mappings = {
            "Premier League 2022-2023": {"start_year": 2022, "end_year": 2023},
            "Serie A 2022-2023": {"start_year": 2022, "end_year": 2023},
            "La Liga 2022-2023": {"start_year": 2022, "end_year": 2023},
            "Bundesliga 2022-2023": {"start_year": 2022, "end_year": 2023},
            "Champions League 2022-2023": {"start_year": 2022, "end_year": 2023},
            "La Liga 2 2022-2023": {"start_year": 2022, "end_year": 2023},
            "Eredivisie 2022-2023": {"start_year": 2022, "end_year": 2023},
            "Jupiler Pro League 2022-2023": {"start_year": 2022, "end_year": 2023},
            "Chinese Super League 2023": {"start_year": 2023, "end_year": 2023},
            "K-League 1 2023": {"start_year": 2023, "end_year": 2023}
        }
        
        if league not in season_mappings:
            print(f"Warning: League '{league}' not found in mappings")
            return None
        
        season_info = season_mappings[league]
        
        # Calendar year leagues (2023 only)
        if season_info["start_year"] == season_info["end_year"]:
            return season_info["start_year"]
        
        # Split season leagues (2022-2023)
        # Aug, Sep, Oct, Nov, Dec = start year (2022)
        # Jan, Feb, Mar, Apr, May, Jun, Jul = end year (2023)
        late_year_months = ['Aug', 'Sep', 'Oct', 'Nov', 'Dec']
        if month in late_year_months:
            return season_info["start_year"]  # 2022
        else:
            return season_info["end_year"]    # 2023
    
    # Apply the function
    df_match['year'] = df_match.apply(get_year, axis=1)
    
    return df_match

# Apply the function
df_match = assign_year_from_month_and_league(df_match)

# Display results by league to verify
print("Year assignment results:")
for league in df_match['league'].unique()[:5]:  # Show first 5 leagues
    league_data = df_match[df_match['league'] == league]
    print(f"\n{league}:")
    month_year_counts = league_data.groupby(['month', 'year']).size()
    for (month, year), count in month_year_counts.head(6).items():
        print(f"  {month} {year}: {count} matches")

# Check for any missing years
missing_years = df_match[df_match['year'].isna()]
if not missing_years.empty:
    print(f"\nWarning: {len(missing_years)} matches couldn't assign years")
    print(missing_years[['league', 'month']].value_counts().head())
else:
    print(f"\n✓ Successfully assigned years to all {len(df_match)} matches")

Year assignment results:

Premier League 2022-2023:
  Apr 2023.0: 61 matches
  Aug 2022.0: 44 matches
  Dec 2022.0: 13 matches
  Feb 2023.0: 40 matches
  Jan 2023.0: 40 matches
  Mar 2023.0: 31 matches

Serie A 2022-2023:
  Apr 2023.0: 48 matches
  Aug 2022.0: 33 matches
  Feb 2023.0: 38 matches
  Jan 2023.0: 50 matches
  Jun 2023.0: 11 matches
  Mar 2023.0: 32 matches

La Liga 2022-2023:
  Apr 2023.0: 56 matches
  Aug 2022.0: 30 matches
  Dec 2022.0: 8 matches
  Feb 2023.0: 42 matches
  Jan 2023.0: 40 matches
  Jun 2023.0: 10 matches

Bundesliga 2022-2023:
  Apr 2023.0: 44 matches
  Aug 2022.0: 36 matches
  Feb 2023.0: 36 matches
  Jan 2023.0: 27 matches
  Jun 2023.0: 2 matches
  Mar 2023.0: 27 matches

Champions League 2022-2023:
  Apr 2023.0: 8 matches
  Aug 2022.0: 32 matches
  Feb 2023.0: 8 matches
  Jul 2023.0: 54 matches
  Jun 2023.0: 4 matches
  Mar 2023.0: 8 matches

Series([], Name: count, dtype: int64)


# Name convert

In [54]:
def clean_team_names(team_name):
    """Clean team names by removing contract dates and standardizing names"""
    if pd.isna(team_name):
        return team_name
    
    # Remove contract date suffixes first
    team_name = re.sub(r'(Jun|Dec|Jan|Feb|Mar|Apr|May|Jul|Aug|Sep|Oct|Nov)\s+\d{1,2},?', '', str(team_name))
    
    # Remove common prefixes and suffixes
    team_name = re.sub(r'^(FC|CF|AC|AS|SC|RC|RCD|CD|SD|UD|VfL|VfB|1\.\s?FC|KRC|KAA|KV|KVC|RSC|RFC|Real|SV)\s+', '', team_name)
    team_name = re.sub(r'\s+(FC|CF|AC|AS|SC|RC|RCD|CD|SD|UD|VfL|VfB|City)$', '', team_name)
    
    # Comprehensive bidirectional mapping
    name_mapping = {
        # Fixed Spanish teams
        'US Salernitana': 'Salernitana',
        'Salernitana': 'Salernitana',
        'Real Betis Balompié': 'Betis',
        'Betis Balompié': 'Betis',
        'Betis': 'Betis',
        'CA Osasuna': 'Osasuna',
        'Osasuna': 'Osasuna',
        'Cádiz': 'Cadiz',
        'Cadiz': 'Cadiz',
        'Deportivo Alavés': 'Alaves',
        'Alaves': 'Alaves',
        'Albacete Balompié': 'Albacete',
        'Albacete': 'Albacete',
        'Real Sporting de Gijón': 'Sporting de Gijón',
        'Sporting de Gijón': 'Sporting de Gijón',
        'Cartagena': 'Cartagena',
        'Cartagena SAD': 'Cartagena',
        'Villarreal CF B': 'Villarreal B',
        'Villarreal B': 'Villarreal B',
        
        # Fixed German teams
        'Bayer 04 Leverkusen': 'Bayer Leverkusen',
        'Bayer Leverkusen': 'Bayer Leverkusen',
        '1. FSV Mainz': 'Mainz',
        'Mainz': 'Mainz',
        'Hertha BSC': 'Hertha Berlin',
        'Hertha Berlin': 'Hertha Berlin',
        'TSG': 'Hoffenheim',
        'Hoffenheim': 'Hoffenheim',
        'Schalke': 'Schalke',
        
        # Fixed Dutch teams
        'Go Ahead Eagles': 'G.A. Eagles',
        'G.A. Eagles': 'G.A. Eagles',
        'NEC Nijmegen': 'Nijmegen',
        'Nijmegen': 'Nijmegen',
        'RKC Waalwijk': 'Waalwijk',
        'Waalwijk': 'Waalwijk',
        'Fortuna Sittard': 'Sittard',
        'Sittard': 'Sittard',
        'Almere': 'Almere',
        'Eindhoven FC': 'Eindhoven',
        
        # Fixed Belgian teams
        'Union Saint-Gilloise': 'Royale Union SG',
        'Royale Union SG': 'Royale Union SG',
        'Standard de Liège': 'St. Liege',
        'St. Liege': 'St. Liege',
        'Royal Charleroi Sporting Club': 'Charleroi',
        'Charleroi': 'Charleroi',
        'Oud-Heverlee Leuven': 'Leuven',
        'Leuven': 'Leuven',
        'SV Zulte Waregem': 'Zulte Waregem',
        'Zulte Waregem': 'Zulte Waregem',
        'Sint-Truidense VV': 'St. Truiden',
        'St. Truiden': 'St. Truiden',
        
        # Fixed Chinese teams
        'Dalian Professional': 'Dalian Pro',
        'Dalian Pro': 'Dalian Pro',
        
        # Fixed K-League teams
        'Suwon Samsung Bluewings': 'Suwon Bluewings',
        'Suwon Bluewings': 'Suwon Bluewings',
        'Pohang Steelers': 'Pohang',
        'Pohang': 'Pohang',
        'Jeonbuk Hyundai Motors': 'Jeonbuk',
        'Jeonbuk': 'Jeonbuk',
        
        # English teams (shortened names)
        'Birmingham': 'Birmingham',
        'Bristol': 'Bristol',
        'Cardiff': 'Cardiff',
        'Coventry': 'Coventry',
        'Hull': 'Hull',
        'Norwich': 'Norwich',
        'Preston North End': 'Preston',
        'Queens Park Rangers': 'QPR',
        'Reading': 'Reading',
        'Stoke': 'Stoke',
        'Swansea': 'Swansea',
        'Watford': 'Watford',
        
        # Keep common name variations
        'Manchester Utd': 'Manchester United',
        'Brighton': 'Brighton & Hove Albion',
        'Newcastle': 'Newcastle United',
        'West Ham': 'West Ham United',
        'Nottingham': 'Nottingham Forest',
        'Leicester': 'Leicester City',
        'Leeds': 'Leeds United',
        'Tottenham': 'Tottenham Hotspur',
        'Wolves': 'Wolverhampton Wanderers',
    }
    
    # Clean and strip
    team_name = team_name.strip()
    
    # Apply mapping
    if team_name in name_mapping:
        return name_mapping[team_name]
    
    return team_name

# Apply cleaning to both datasets
df['team'] = df['team'].apply(clean_team_names)
df_match['home_team'] = df_match['home_team'].apply(clean_team_names)
df_match['away_team'] = df_match['away_team'].apply(clean_team_names)

In [ ]:
# Determine match winner
def determine_winner(row):
    """
    Determine match winner based on home and away scores
    Returns: 1 if home team wins, 0.5 if draw, 0 if away team wins
    """
    home_score = row['home_score']
    away_score = row['away_score']
    
    if pd.isna(home_score) or pd.isna(away_score):
        return None  # Handle missing scores
    
    if home_score > away_score:
        return 1  # Home team wins
    elif home_score == away_score:
        return 0.5  # Draw
    else:
        return 0  # Away team wins

# Apply the function to create winner column
df_match['home_team_win'] = df_match.apply(determine_winner, axis=1)

# Display result distribution
print("Match result distribution:")
result_counts = df_match['home_team_win'].value_counts().sort_index()
for result, count in result_counts.items():
    if result == 1:
        print(f"Home team wins: {count}")
    elif result == 0.5:
        print(f"Draws: {count}")
    elif result == 0:
        print(f"Away team wins: {count}")
    else:
        print(f"Missing data: {count}")

Match result distribution:
Away team wins: 800
Draws: 740
Home team wins: 1280


,home_team,away_team,home_score,away_score,league,month,year,home_team_win
0,Arsenal,Wolverhampton Wanderers,5,0,Premier League 2022-2023,May,2023.0,1.0
1,Aston Villa,Brighton & Hove Albion,2,1,Premier League 2022-2023,May,2023.0,1.0
2,Brentford,Manchester,1,0,Premier League 2022-2023,May,2023.0,1.0
3,Chelsea,Newcastle United,1,1,Premier League 2022-2023,May,2023.0,0.5
4,Crystal Palace,Nottingham Forest,1,1,Premier League 2022-2023,May,2023.0,0.5
...,...,...,...,...,...,...,...,...
2815,Jeju SK,Suwon,0,0,K-League 1 2023,Feb,2023.0,0.5
2816,Pohang,Daegu,3,2,K-League 1 2023,Feb,2023.0,1.0
2817,Seoul,Incheon,2,1,K-League 1 2023,Feb,2023.0,1.0
2818,Suwon Bluewings,Gwangju,0,1,K-League 1 2023,Feb,2023.0,0.0


In [58]:
df_match.to_csv("../../data/final_train_data/match.csv")